# 1. CARGA Y VISIÓN GENERAL

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)

# Cargar dataset
df = pd.read_csv('data/train.csv')
print('Shape:', df.shape)
display(df.head())

# Eliminar columnas ID si existen
df.drop([col for col in ['Id', 'index'] if col in df.columns], axis=1, inplace=True)

# 2. ANÁLISIS DEL TARGET (SalePrice)

In [ ]:
target = 'SalePrice'
# Histograma original
df[target].hist(bins=40, color='steelblue', edgecolor='black')
plt.title('Distribución Original del Target (SalePrice)')
plt.xlabel('SalePrice')
plt.ylabel('Frecuencia')
plt.show()

# Chequear si es continuo o categórico
target_unique = df[target].nunique()
if target_unique < 30:
    print('El target parece categórico: valores únicos =', target_unique)
    vc = df[target].value_counts()
    sns.barplot(x=vc.index, y=vc.values, palette='viridis')
    plt.title('Balance de Clases en Target')
    plt.ylabel('Frecuencia')
    plt.show()
else:
    skew = df[target].skew() # ~1.88, sesgado
    print(f'Sesgo original: {skew:.2f}')
    # Si es sesgado, intentar log
    df['log_'+target] = np.log(df[target])
    df['log_'+target].hist(bins=40, color='forestgreen', edgecolor='black')
    plt.title('Distribución de log(SalePrice)')
    plt.xlabel('log(SalePrice)')
    plt.ylabel('Frecuencia')
    plt.show()
    print('Sesgo luego de log:', df['log_'+target].skew())

# 3. TIPOS DE VARIABLES

In [ ]:
# Categóricas reales
cat_vars = df.select_dtypes(include=['object']).columns.tolist()
# Numéricas disfrazadas de categóricas (según descripción y valores únicos)
cat_like = []
for col in ['MSSubClass']:
    if col in df.columns:
        df[col] = df[col].astype(str)
        cat_vars.append(col)
        cat_like.append(col)
# Variables numéricas
num_vars = df.select_dtypes(include=[np.number]).columns.drop(target).tolist()
num_vars = [col for col in num_vars if col not in cat_like]
print(f'Categóricas: {len(cat_vars)} -->\n', cat_vars)
print(f'Numéricas: {len(num_vars)} -->\n', num_vars)

# 4. MISSING VALUES

In [ ]:
na_pct = df.isna().mean().sort_values(ascending=False)
na_cols = na_pct[na_pct>0]
print('Variables con missing values:', list(na_cols.index))
display(na_pct[na_pct>0])
# Barplot
plt.figure(figsize=(10,6))
sns.barplot(x=na_pct[na_pct>0]*100, y=na_pct[na_pct>0].index, palette='magma')
plt.title('Porcentaje de Missing Values por Variable')
plt.xlabel('% Missing')
plt.ylabel('Variable')
plt.show()
# Separar categóricas y numéricas con NA
cat_na = [c for c in na_cols.index if c in cat_vars]
num_na = [c for c in na_cols.index if c in num_vars]

# Relación missing vs target
for col in na_cols.index:
    df[col+'_na'] = df[col].isna().astype(int)
    plt.figure(figsize=(4,2))
    sns.barplot(x=col+'_na', y=target, data=df)
    plt.title(f'Media del target por missing en {col}')
    plt.ylabel('SalePrice')
    plt.xticks([0,1], ['No Missing','Missing'])
    plt.show()

# 5. VARIABLES NUMÉRICAS

## 5a. VARIABLES TEMPORALES

In [ ]:
time_keywords = ['Yr', 'Year']
temp_vars = [v for v in num_vars if any(tok in v for tok in time_keywords)]
print('Variables temporales:', temp_vars)
reference_year = df['YrSold'] if 'YrSold' in df.columns else None
for v in temp_vars:
    plt.figure(figsize=(8,3))
    med = df.groupby(v)[target].median()
    plt.plot(med.index, med.values)
    plt.title(f'Mediana del target vs {v}')
    plt.ylabel('Mediana SalePrice')
    plt.xlabel(v)
    plt.show()
    if reference_year is not None and v != 'YrSold':
        años_transcurridos = reference_year - df[v]
        plt.scatter(años_transcurridos, df[target], alpha=0.3)
        plt.title(f'Años transcurridos (YrSold-{v}) vs SalePrice')
        plt.xlabel(f'Años transcurridos desde {v}')
        plt.ylabel('SalePrice')
        plt.show()

## 5b. VARIABLES DISCRETAS

In [ ]:
discrete_vars = [v for v in num_vars if df[v].nunique() < 20 and v not in temp_vars]
for v in discrete_vars:
    plt.figure(figsize=(7,3))
    sns.boxplot(x=v, y=target, data=df, showmeans=True)
    sns.stripplot(x=v, y=target, data=df, alpha=0.3, jitter=True, color='k')
    plt.title(f'{v} vs SalePrice')
    plt.show()

## 5c. VARIABLES CONTINUAS

In [ ]:
# Usar skew > 0.75 definida por profiler (variables candidatas a transformaciones)
skewed = ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'SalePrice', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea', 'BsmtUnfSF', '2ndFlrSF']
continuous_vars = [v for v in num_vars if v not in discrete_vars+temp_vars]

for v in continuous_vars:
    plt.figure(figsize=(7,3))
    sns.histplot(df[v].dropna(), kde=True, bins=40, color='steelblue')
    plt.title(f'Distribución Original: {v}')
    plt.show()
    # Yeo-Johnson
    from sklearn.preprocessing import PowerTransformer
    transformer = PowerTransformer(method='yeo-johnson')
    try:
        vals_t = df[v].dropna().values.reshape(-1,1)
        yt = transformer.fit_transform(vals_t).flatten()
        plt.figure(figsize=(7,3))
        sns.histplot(yt, kde=True, bins=40, color='goldenrod')
        plt.title(f'Yeo-Johnson: {v}')
        plt.show()
        # Scatter log(target)
        fig, (ax1, ax2) = plt.subplots(1,2, figsize=(10,3))
        ax1.scatter(df[v], df['log_'+target], alpha=0.25)
        ax1.set_title(f'{v} vs log(SalePrice)')
        ax2.scatter(yt, df['log_'+target'].dropna(), alpha=0.25)
        ax2.set_title(f'Yeo-Johnson({v}) vs log(SalePrice)')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Error en Yeo-Johnson para {v}: {e}')
    # Transformación log si no tiene ceros
    if (df[v] > 0).all():
        vals_ln = np.log(df[v].dropna())
        plt.figure(figsize=(7,3))
        sns.histplot(vals_ln, kde=True, bins=40, color='seagreen')
        plt.title(f'log({v})')
        plt.show()

## 5d. VARIABLES SESGADAS / DOMINADAS POR CEROS

In [ ]:
# Usar exclusivas del profiler (>50% ceros)
zero_dominated = ['MasVnrArea', 'BsmtFinSF2', '2ndFlrSF', 'LowQualFinSF', 'BsmtFullBath', 'BsmtHalfBath', 'HalfBath', 'WoodDeckSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal']
for v in zero_dominated:
    if v in df.columns:
        df[v+'_bin'] = (df[v] > 0).astype(int)
        sns.barplot(x=df[v+'_bin'], y=df[target])
        plt.title(f'Media del target por {v} binarizado')
        plt.xlabel(f'{v}==0 / !=0')
        plt.ylabel('SalePrice')
        plt.show()

# 6. VARIABLES CATEGÓRICAS

## 6a. CARDINALIDAD

In [ ]:
cat_unique = df[cat_vars].nunique().sort_values(ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(y=cat_unique.index, x=cat_unique.values)
plt.title('Número de categorías únicas por variable categórica')
plt.xlabel('Nº Categorías')
plt.ylabel('Variable')
plt.show()
# Alta cardinalidad: más de 10
high_card = cat_unique[cat_unique>10]
print('Alta cardinalidad (>10):', list(high_card.index))

## 6b. VARIABLES ORDINALES DE CALIDAD

In [ ]:
ordinal_grupos = {
    'ExterQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'ExterCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'HeatingQC': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'KitchenQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'FireplaceQu': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'GarageQual': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'GarageCond': {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1}
}
for var,mapping in ordinal_grupos.items():
    if var in df.columns:
        df[var+'_num'] = df[var].map(mapping)
        sns.boxplot(x=df[var+'_num'], y=df[target])
        plt.title(f'{var} mapeada vs SalePrice')
        plt.show()

## 6c. RARE LABELS (frecuencia <1%)

In [ ]:
for v in cat_vars:
    vc = df[v].value_counts(normalize=True)
    rare = vc[vc<0.01]
    if len(rare)>0:
        print(f'{v}: {list(rare.index)}')

## 6d. CATEGÓRICAS NOMINALES vs TARGET

In [ ]:
# Excluir ordinales y mappings
desde_ord = set(ordinal_grupos.keys())
nominales = [v for v in cat_vars if v not in desde_ord and v!='MSSubClass']
for v in nominales:
    if df[v].nunique() <= 15: # solo si es manejable
        plt.figure(figsize=(10,3))
        sns.boxplot(x=df[v], y=df[target])
        sns.stripplot(x=df[v], y=df[target], alpha=0.3, color='k', jitter=True)
        plt.title(f'{v} vs SalePrice')
        plt.xticks(rotation=45)
        plt.show()

# 7. CONCLUSIONES

```markdown
## Variables que necesitan imputación:
- Numéricas con missing: LotFrontage, GarageYrBlt, MasVnrArea
- Categóricas con missing: PoolQC, MiscFeature, Alley, Fence, MasVnrType, FireplaceQu, GarageType, GarageFinish, GarageQual, GarageCond, BsmtFinType2, BsmtExposure, BsmtFinType1, BsmtCond, BsmtQual, Electrical

## Variables temporales detectadas: YearBuilt, YearRemodAdd, GarageYrBlt, MoSold, YrSold
- Variable de referencia: YrSold

## Variables continuas candidatas a log y a Yeo-Johnson:
- LotArea, GrLivArea, 1stFlrSF, TotalBsmtSF, SalePrice, WoodDeckSF, OpenPorchSF, MasVnrArea, LotFrontage (todas con skew alto; revisar ceros para log)

## Variables sesgadas candidatas a binarización:
- MasVnrArea, BsmtFinSF2, 2ndFlrSF, LowQualFinSF, BsmtFullBath, BsmtHalfBath, HalfBath, WoodDeckSF, EnclosedPorch, 3SsnPorch, ScreenPorch, PoolArea, MiscVal

## Variables ordinales y mappings sugeridos:
- ExterQual, ExterCond, BsmtQual, BsmtCond, HeatingQC, KitchenQual, FireplaceQu, GarageQual, GarageCond (mapping Ex=5, Gd=4, TA=3, Fa=2, Po=1)

## Variables con rare labels candidatas a RareLabelEncoder:
- Revisar output de rare labels detectados en 6c

## Variables a eliminar:
- Id (ya eliminada)
```